## Importação de Bibliotecas

In [86]:
import gzip
import random
import math
from datetime import datetime


## Definição de funções

### Função de carga de dataset em dicionário

arquivo: Recebe o nome do arquivo .tsp.gz que será carregado

- Converte o arquivo .tsp.gz em uma varíavel String (str); 
- Realiza os tratamentos identificando as linhas que contém os dados de cidade (desconsiderando todas as linhas que não possuem dados úteis sobre a coordenada das cidades); 
- Itera sobre todas as linhas de cidade, adicionando as coordenadas da cidade em um dicionário;
- Retorna o dicionário inteiro com as chaves sendo o ID da cidade (str de 1 até n);

In [87]:
def carregar_dataset_dicionario(arquivo: str) -> dict:
    with gzip.open(CAMINHO + arquivo, 'rt', encoding='utf-8') as arq:
        arquivo_completo_str = arq.readlines()

        for linha in arquivo_completo_str:
            linha = linha

            if linha.split()[0] in LINHAS_DESCONSIDERAR:
                continue

            if linha.strip() == 'EOF' or not linha:
                break

            cidade = linha.split()
            cidades[cidade[0]] = (cidade[1], cidade[2])
    
    return cidades

### Função de gerador de individuos

cidades: Recebe um dicionário contendo o dataset com todas as cidades e suas coordenadas para teste.

qtd_individuos: Recebe a quantidade de indivíduos que deseja ser gerado.

- Contabiliza a quantidade de cidades do dataset (que corresponde ao cromossomo deste caso) e as armazena em uma variável; 
- Cria uma lista vazia de individuos criados;
- Itera sobre a quantidade de individuos que se deseja serem criados (de acordo com a quantidade recebida nos parâmetros);
- Percorre a quantidade de cromossomos (cidades) para carregar TODAS em um indivíduo;
- Seleciona um cromossomo aleatóriamente;
- Valida se este cromossomo já foi visitado antes. Caso tenha sido, seleciona outro até resultar em um não selecionado;
- Adicionada o cromossomo escolhido no conjunto de visitados;
- Adiciona o cromossomo ao indivíduo;
- Ao gerar o indivíduo com todos os cromossomos, reseta a variável e cria outro indivíduo;
- Retorna uma lista de indivíduos gerados aleatoriamente.

In [88]:
def gerador_individuos(cidades: dict, qtd_individuos: int) -> list:
    qtd_cromossomos = len(cidades)
    individuos = list()

    for i in range(qtd_individuos):
        cidades_visitadas = set()
        individuo = list()

        for _ in range(qtd_cromossomos):
            cromossomo_selecionado = str(random.randint(1, qtd_cromossomos))

            while cromossomo_selecionado in cidades_visitadas:
                cromossomo_selecionado = str(random.randint(1, qtd_cromossomos))

            cidades_visitadas.add(cromossomo_selecionado)

            individuo.append(cromossomo_selecionado)

        individuos.append(individuo)
    
    return individuos

### Função de Calcular Distância

dataset: Recebe o dicionário com as coordenadas de cada cidade do dataset.

cromossomo: Recebe uma lista com o cromossomo do indivíduo que será calculada a aptidão.

- Declara as variáveis iniciais;
- Percorre a quantidade de alelos do cromossomo;
- Verifica a condição de parada onde, validando se o alelo seguinte ainda existe;
- Coleta a cidade atual aonde o caixeiro está e a próxima cidade que irá em direção;
- Realiza os cálculos da distância x atual até x próxima e y atual até y próxima;
- Realiza o cálculo da distância entre uma cidade e outra;
- Soma a distância entre as cidades à variável inicial de distância total percorrida;
- Realiza a soma de distâncias entre uma cidade e outra até acabarem as cidades (alelos).

In [89]:
def calcular_distancia(dataset, cromossomo: list):
    qtd_alelos = len(cromossomo)
    distancia_total = 0
    for i in range(qtd_alelos):
        if i+1 == qtd_alelos:
            break
        
        cidade_atual = dataset[cromossomo[i]]
        proxima_cidade = dataset[cromossomo[i + 1]]

        dx = float(proxima_cidade[0]) - float(cidade_atual[0])
        dy = float(proxima_cidade[1]) - float(cidade_atual[1])

        d = round(math.sqrt((dx)**2 + (dy)**2))
        distancia_total += d

    return distancia_total

### Função de aptidão

dataset: Recebe o dicionário com as coordenadas de cada cidade do dataset.

cromossomo: Recebe uma lista com o cromossomo do indivíduo que será calculada a aptidão.

- Realiza a chamada da função de cálculo de distância;
- Retorna o resultado do cálculo da fórmula de aptidão.

In [90]:
def calcular_aptidao(dataset: dict, cromossomo: list) -> float:
    distancia_total = calcular_distancia(dataset, cromossomo)

    return 1 / (1 + distancia_total)

### Função de Torneio

dataset: Recebe o dicionário com as coordenadas de cada cidade do dataset.

populacao: Lista contendo outras listas que representam os indivíduos da população.

- Coleta dois indivíduos aleatórios da população;
- Verifica se são diferentes. Somente seguirá o código se forem diferentes;
- Calcula a aptidão de cada um;
- Retorna o indivíduo com maior aptidão.

In [91]:
def torneio(dataset: dict, populacao: list) -> list:
    tamanho_populacao = len(populacao)

    indice_i1 = random.randint(0, tamanho_populacao-1)
    indice_i2 = random.randint(0, tamanho_populacao-1)

    while indice_i1 == indice_i2:
        indice_i2 = random.randint(0, tamanho_populacao-1)

    i1 = populacao[indice_i1]
    i2 = populacao[indice_i2]

    aptidao_i1 = calcular_aptidao(dataset, i1)
    aptidao_i2 = calcular_aptidao(dataset, i2)

    return i1.copy() if aptidao_i1 > aptidao_i2 else i2.copy()

### Função de Crossover

i1: Indivíduo 1

i2: Indivíduo 2

- Verifica se o tamanho do cromossomo de ambos os indivíduos são diferentes. Se for, levanta um erro que interrompe o código;
- Sorteia dois pontos aleatórios do indivíduo que serão permutados;
- Retorna os resultados de duas chamadas da função auxiliar_crossover, onde o que muda em cada chamada é a ordem de indivíduos.

In [92]:
def crossover(i1: list, i2: list) -> tuple:
    tamanho_individuo = len(i1)

    if len(i1) != len(i2):
        raise("ERRO: Tamanho de cromossomos do indivíduo estão diferentes na chamada de função de Crossover")

    ponto1, ponto2 = sorted(random.sample(range(tamanho_individuo), 2))

    return auxiliar_crossover(i1, i2, ponto1, ponto2, tamanho_individuo), auxiliar_crossover(i2, i1, ponto1, ponto2, tamanho_individuo)
    


### Função Auxiliar Crossover

i1: Recebe uma lista contendo o cromossomo do primeiro indivíduo pai.

i2: Recebe uma lista contendo o cromossomo do segundo indivíduo pai.

ponto1: Recebe um inteiro indicando o índice inicial do corte no cromossomo.

ponto2: Recebe um inteiro indicando o índice final do corte no cromossomo.

tamanho: Recebe um inteiro com a quantidade total de alelos do cromossomo.

- Cria uma lista preenchida com valores nulos do mesmo tamanho do cromossomo para armazenar o filho;
- Copia o segmento entre ponto1 e ponto2 do cromossomo i1 para as mesmas posições no filho;
- Armazena os alelos já inseridos no filho em um conjunto (set) para evitar a repetição de cidades;
- Reorganiza o cromossomo i2 de forma circular a partir de ponto2, concatenando a parte final com a parte inicial;
- Filtra os alelos de i2, mantendo apenas os genes que ainda não estão presentes no filho;
- Declara uma variável de índice auxiliar inicializada em zero para percorrer os genes restantes;
- Percorre circularmente todas as posições do filho iniciando em ponto2;
- Verifica se a posição atual está vazia (None), preenche com o próximo gene restante disponível e incrementa o índice auxiliar;
- Retorna a lista contendo o cromossomo completo e válido do indivíduo filho gerado.

In [93]:
def auxiliar_crossover(i1: list, i2: list, ponto1: int, ponto2: int, tamanho: int) -> list:
    filho = [None] * tamanho
    filho[ponto1:ponto2] = i1[ponto1:ponto2]

    genes_presentes = set(filho[ponto1:ponto2])
    p2_circular = i2[ponto2:] + i2[:ponto2]
    genes_restantes = [gene for gene in p2_circular if gene not in genes_presentes]

    idx_restante = 0
    for i in range(tamanho):
        pos = (ponto2 + i) % tamanho
        if filho[pos] is None:
            filho[pos] = genes_restantes[idx_restante]
            idx_restante += 1

    return filho

### Função de Mutação

cromossomo: Recebe uma lista com o cromossomo do indivíduo que será calculada a aptidão.

- Cria uma cópia do cromossomo original (recebido no parâmetro);
- Coleta aleatóriamente dois pontos únicos desse cromossomo;
- Realiza a permutação do valor dos dois índices;
- Retorna o cromossomo mutado.  

In [94]:
def mutacao(cromossomo: list) -> list:
    individuo = cromossomo.copy()
    
    idx1, idx2 = random.sample(range(len(individuo)), 2)
    individuo[idx1], individuo[idx2] = individuo[idx2], individuo[idx1]

    return individuo

## Leitura e preparação de Datasets

In [95]:
CAMINHO = 'utils/base_dados_mochila/'
ARQUIVOS = ['berlin52.tsp.gz', 'eil101.tsp.gz', 'kroA100.tsp.gz', 'rat99.tsp.gz', 'st70.tsp.gz']
LINHAS_DESCONSIDERAR = ['NAME:', 'TYPE:', 'COMMENT:', 'DIMENSION:', 'EDGE_WEIGHT_TYPE:', 'NODE_COORD_SECTION', 'NAME', 'TYPE', 'COMMENT', 'DIMENSION', 'EDGE_WEIGHT_TYPE', 'NODE_COORD_SECTION']

datasets_inicial = {}
for arquivo in ARQUIVOS:
    cidades = {}
    cidades = carregar_dataset_dicionario(arquivo)
    datasets_inicial[arquivo] = cidades

for dataset in datasets_inicial.values():
    print("Qtd de cidades:", len(dataset))



Qtd de cidades: 52
Qtd de cidades: 101
Qtd de cidades: 100
Qtd de cidades: 99
Qtd de cidades: 70


## Execução Principal do Algoritmo Genético

In [96]:
QTD_GERACOES = 500
TAMANHO_POPULACAO = 50
PROBABILIDADE_CROSSOVER = 0.85
PROBABILIDADE_MUTACAO = 0.10
data_execucao = {}
individuo_otimo = {}

for chave, dataset in datasets_inicial.items():
    populacao = gerador_individuos(dataset, TAMANHO_POPULACAO)
    inicio_execucao = datetime.now()

    for i in range(QTD_GERACOES):
        individuo_elite = max(populacao, key=lambda cromossomo: calcular_aptidao(dataset, cromossomo)).copy()
        nova_populacao = [individuo_elite.copy()]

        while len(nova_populacao) < len(populacao):
            nova_populacao.append(torneio(dataset, populacao).copy())

        ordem_sorteada = random.sample(range(len(populacao)), k=len(populacao))
        for j in range(0, len(ordem_sorteada), 2):
            indice_1 = nova_populacao[ordem_sorteada[j]]
            indice_2 = nova_populacao[ordem_sorteada[j+1]]

            if random.random() < PROBABILIDADE_CROSSOVER:
                nova_populacao[ordem_sorteada[j]], nova_populacao[ordem_sorteada[j+1]] = crossover(indice_1, indice_2)

            if random.random() < PROBABILIDADE_MUTACAO:
                nova_populacao[ordem_sorteada[j]] = mutacao(nova_populacao[ordem_sorteada[j]])
                nova_populacao[ordem_sorteada[j+1]] = mutacao(nova_populacao[ordem_sorteada[j+1]])
                
        populacao = nova_populacao.copy()
    
    fim_execucao = datetime.now()
    data_execucao[chave] = (inicio_execucao, fim_execucao)
    individuo_otimo[chave] = max(populacao, key=lambda cromossomo: calcular_aptidao(dataset, cromossomo)).copy()

## Apresentação dos Resultados Finais

In [97]:
print("========== RESULTADOS DA EXECUÇÃO ==========")
print("Tamanho da população: ", TAMANHO_POPULACAO)
print("Qtd. de Gerações: ", QTD_GERACOES)
print("Probabilidade de Crossover: ", PROBABILIDADE_CROSSOVER)
print("Probabilidade de Mutação: ", PROBABILIDADE_MUTACAO)
print("--------------------------------------------")

for chave, dataset in datasets_inicial.items():
    print("\nDataset analisado: ", chave)
    print("Início da execução: ", data_execucao[chave][0])
    print("Fim da execução: ", data_execucao[chave][1])
    print("Duração da execução: ", data_execucao[chave][1] - data_execucao[chave][0])
    print("Cromossomo ótimo: ", individuo_otimo[chave])
    print("Distância: ", calcular_distancia(dataset, individuo_otimo[chave]))
    print("Aptidão: ", calcular_aptidao(dataset, individuo_otimo[chave]))

========== RESULTADOS DA EXECUÇÃO ==========
Tamanho da população:  50
Qtd. de Gerações:  500
Probabilidade de Crossover:  0.85
Probabilidade de Mutação:  0.1
--------------------------------------------

Dataset analisado:  berlin52.tsp.gz
Início da execução:  2026-11-26 17:44:14.200952
Fim da execução:  2026-11-26 17:44:20.482149
Duração da execução:  0:00:06.281197
Cromossomo ótimo:  ['27', '12', '51', '11', '52', '13', '14', '26', '28', '33', '43', '36', '22', '42', '2', '7', '3', '8', '10', '9', '41', '40', '6', '34', '39', '35', '44', '16', '29', '47', '4', '15', '5', '46', '23', '20', '50', '30', '21', '31', '17', '18', '19', '45', '37', '24', '38', '32', '49', '1', '48', '25']
Distância:  11873
Aptidão:  8.421761832575375e-05

Dataset analisado:  eil101.tsp.gz
Início da execução:  2026-11-26 17:44:20.498811
Fim da execução:  2026-11-26 17:44:32.476851
Duração da execução:  0:00:11.978040
Cromossomo ótimo:  ['87', '2', '43', '5', '60', '89', '18', '7', '70', '1', '66', '88', '58